# Day 11 — Threshold Optimization & Risk Banding

**Goal:** Convert raw default probabilities into actionable credit decisions. Find the optimal classification threshold, quantify business cost at each threshold, and define three risk tiers for automated routing.

## Sections
1. Setup & Imports
2. Data Load + Feature Engineering
3. Train / Test Split
4. Train XGBoost Champion Model
5. Task 11.1 — Precision-Recall Curve
6. Task 11.2 — Threshold Cost Analysis
7. Task 11.3 — Risk Band Definition & Visualization
8. Final Decision & Policy Reference

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    roc_auc_score, precision_score, recall_score,
    f1_score, confusion_matrix,
)
from xgboost import XGBClassifier

from src.features.engineer import engineer_features

plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_STATE = 42
FIGURES = PROJECT_ROOT / 'reports/figures'

## 2. Data Load + Feature Engineering

In [ ]:
for fname in ['credit_default_eda_ready.parquet',
              'credit_default_validated.parquet',
              'credit_default_cleaned.parquet']:
    path = PROJECT_ROOT / 'data/processed' / fname
    if path.exists():
        df_raw = pd.read_parquet(path)
        print(f'Loaded: {fname}  |  Shape: {df_raw.shape}')
        break

df_raw = df_raw.drop(columns=[c for c in ['edu_label', 'mar_label', 'age_group']
                               if c in df_raw.columns])
df = engineer_features(df_raw)
print(f'After engineering: {df.shape}  (+{df.shape[1] - df_raw.shape[1]} features)')

## 3. Train / Test Split

Same `random_state=42` and `stratify=y` as Day 10 — identical split guarantees fair comparison.

In [ ]:
X = df.drop(columns=['default', 'risk_segment'])
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train default rate: {y_train.mean():.3f}')
print(f'scale_pos_weight:   {scale_pos_weight:.2f}')

## 4. Train XGBoost Champion Model

In [ ]:
xgb = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
)
xgb.fit(X_train, y_train)

y_probs = xgb.predict_proba(X_test)[:, 1]
y_preds_50 = xgb.predict(X_test)

auc = roc_auc_score(y_test, y_probs)
print(f'XGBoost AUC-ROC : {auc:.4f}  |  Gini: {2 * auc - 1:.4f}')
print(f'Default t=0.50  — Precision: {precision_score(y_test, y_preds_50):.4f}'
      f'  Recall: {recall_score(y_test, y_preds_50):.4f}')

## 5. Task 11.1 — Precision-Recall Curve

The **Precision-Recall curve** is more informative than ROC for imbalanced credit data because it focuses on the minority class (defaulters) rather than the majority.

- **Left panel:** P-R trade-off space — the red dot marks the threshold that maximises F1.
- **Right panel:** How Precision, Recall, and F1 move as the threshold changes.   The vertical lines show key operating points.

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_probs)
ap = average_precision_score(y_test, y_probs)

f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9)
best_idx       = f1_scores.argmax()
best_threshold = thresholds[best_idx]
best_precision = precision[best_idx]
best_recall    = recall[best_idx]
best_f1        = f1_scores[best_idx]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# — Left: P-R space —
ax = axes[0]
ax.plot(recall, precision, lw=2, color='#2196F3', label=f'XGBoost (AP={ap:.3f})')
ax.scatter(best_recall, best_precision, color='red', zorder=5, s=120,
           label=f'Max-F1 at t={best_threshold:.2f}')
ax.axvline(x=0.70, color='orange', linestyle='--', alpha=0.8, label='Recall=0.70 target')
ax.set_xlabel('Recall (Default Detection Rate)', fontsize=12)
ax.set_ylabel('Precision (1 − False Discovery Rate)', fontsize=12)
ax.set_title('Precision-Recall Curve — XGBoost Champion', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

# — Right: metrics vs threshold —
ax2 = axes[1]
ax2.plot(thresholds, precision[:-1], color='#2196F3', lw=2, label='Precision')
ax2.plot(thresholds, recall[:-1],    color='#FF9800', lw=2, label='Recall')
ax2.plot(thresholds, f1_scores,      color='#4CAF50', lw=2, label='F1')
ax2.axvline(x=0.30, color='red',    linestyle='--', alpha=0.8, label='t=0.30 (conservative)')
ax2.axvline(x=0.50, color='gray',   linestyle='--', alpha=0.8, label='t=0.50 (default)')
ax2.axvline(x=best_threshold, color='purple', linestyle='-.', alpha=0.8,
            label=f't={best_threshold:.2f} (max F1)')
ax2.set_xlabel('Classification Threshold', fontsize=12)
ax2.set_ylabel('Score', fontsize=12)
ax2.set_title('Precision / Recall / F1 vs Threshold', fontsize=13, fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(FIGURES / '19_precision_recall_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Average Precision (AP): {ap:.4f}')
print(f'Max-F1 threshold:       {best_threshold:.3f}')
print(f'  Precision @ max-F1:   {best_precision:.4f}')
print(f'  Recall    @ max-F1:   {best_recall:.4f}')
print(f'  F1        @ max-F1:   {best_f1:.4f}')

## 6. Task 11.2 — Threshold Cost Analysis

**Business cost model:**

| Event | Financial Impact | Rationale |
|-------|-----------------|-----------|
| False Negative (missed default) | **−$5,000** | Average credit-card loss given default |
| False Positive (denied good customer) | **−$500** | Lost annual interest revenue |
| True Negative (approved good customer) | $0 baseline | Revenue captured but not modelled here |
| True Positive (blocked default) | $0 baseline | Loss avoided |

> **Interview tip:** "In a conservative lending environment like Citi, we lower our threshold to 0.3. This increases Recall (catching more defaults) even if it means a slight increase in False Positives. The cost model shows this is the economically rational choice." 

In [ ]:
FN_COST = 5_000   # avg loss on a default ($)
FP_COST =   500   # lost annual interest on a declined good account ($)

thresholds_to_test = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
rows = []

for t in thresholds_to_test:
    y_pred_t = (y_probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()

    total_cost    = fn * FN_COST + fp * FP_COST
    approval_rate = (tn + fn) / len(y_test)
    slip_rate     = fn / (tn + fn) if (tn + fn) > 0 else 0

    rows.append({
        'Threshold':         round(t, 2),
        'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn),
        'Precision':         round(precision_score(y_test, y_pred_t, zero_division=0), 4),
        'Recall':            round(recall_score(y_test, y_pred_t, zero_division=0), 4),
        'F1':                round(f1_score(y_test, y_pred_t, zero_division=0), 4),
        'Approval Rate':     round(approval_rate, 4),
        'Default Slip Rate': round(slip_rate, 4),
        'Total Cost ($)':    f'${total_cost:,.0f}',
        '_cost':             total_cost,
    })

cost_df = pd.DataFrame(rows).set_index('Threshold')

print('=== Threshold Cost Analysis ===')
display_cols = ['TP','FP','TN','FN','Precision','Recall','F1',
                'Approval Rate','Default Slip Rate','Total Cost ($)']
print(cost_df[display_cols].to_string())

best_cost_t = cost_df['_cost'].idxmin()
print(f'\nLowest-cost threshold : {best_cost_t}'
      f'  →  {cost_df.loc[best_cost_t, "Total Cost ($)"]}'
      f'  (Recall={cost_df.loc[best_cost_t, "Recall"]}'
      f'  Precision={cost_df.loc[best_cost_t, "Precision"]})')

In [ ]:
thresh_vals    = cost_df.index.tolist()
costs_k        = cost_df['_cost'].values / 1_000
recalls_plot   = cost_df['Recall'].values
approvals_plot = cost_df['Approval Rate'].values

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Cost curve
ax = axes[0]
ax.plot(thresh_vals, costs_k, lw=2, color='#E53935', marker='o')
for t_line, col, lbl in [(0.30, 'orange', 't=0.30 (conservative)'),
                          (0.50, 'gray',   't=0.50 (default)'),
                          (best_cost_t, 'green', f't={best_cost_t} (min cost)')]:
    ax.axvline(x=t_line, color=col, linestyle='--', alpha=0.85, label=lbl)
ax.set_xlabel('Threshold')
ax.set_ylabel('Total Business Cost ($K)')
ax.set_title('Business Cost vs Threshold', fontweight='bold')
ax.legend(fontsize=9)

# Recall vs threshold
ax = axes[1]
ax.plot(thresh_vals, recalls_plot, lw=2, color='#FF9800', marker='o')
ax.axhline(y=0.70, color='red', linestyle='--', label='Recall=0.70 target')
ax.axvline(x=0.30, color='blue', linestyle='--', alpha=0.6, label='t=0.30')
ax.set_xlabel('Threshold')
ax.set_ylabel('Recall (Default Detection Rate)')
ax.set_title('Recall vs Threshold', fontweight='bold')
ax.legend(fontsize=9)

# Approval rate vs threshold
ax = axes[2]
ax.plot(thresh_vals, approvals_plot, lw=2, color='#2196F3', marker='o')
ax.axvline(x=0.30, color='orange', linestyle='--', alpha=0.85, label='t=0.30')
ax.set_xlabel('Threshold')
ax.set_ylabel('Approval Rate')
ax.set_title('Approval Rate vs Threshold', fontweight='bold')
ax.legend(fontsize=9)

plt.suptitle('Threshold Cost Analysis — XGBoost Champion Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / '20_threshold_cost_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Task 11.3 — Risk Band Definition & Visualization

Three tiers aligned with Citi's automated decisioning workflow:

| Band | Score Range | Action |
|------|-------------|--------|
| **Low Risk** | < 0.20 | Automatic Approval — standard credit limit |
| **Medium Risk** | 0.20 – 0.50 | Manual Review — reduced limit or verification step |
| **High Risk** | > 0.50 | Automatic Decline |

In [ ]:
def assign_risk_band(prob):
    if prob < 0.20:
        return 'Low Risk'
    elif prob < 0.50:
        return 'Medium Risk'
    return 'High Risk'

risk_df = pd.DataFrame({
    'probability': y_probs,
    'actual':      y_test.values,
    'risk_band':   [assign_risk_band(p) for p in y_probs],
})

band_order = ['Low Risk', 'Medium Risk', 'High Risk']
band_summary = (
    risk_df.groupby('risk_band')
    .agg(count=('probability', 'count'),
         default_rate=('actual', 'mean'),
         avg_prob=('probability', 'mean'),
         min_prob=('probability', 'min'),
         max_prob=('probability', 'max'))
    .round(4)
    .loc[band_order]
)
band_summary['pct_of_portfolio'] = (band_summary['count'] / len(risk_df)).round(4)

print('=== Risk Band Summary ===')
print(band_summary[['count', 'pct_of_portfolio', 'default_rate',
                     'avg_prob', 'min_prob', 'max_prob']].to_string())

In [ ]:
COLORS = {'Low Risk': '#4CAF50', 'Medium Risk': '#FF9800', 'High Risk': '#E53935'}
band_colors = [COLORS[b] for b in band_order]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Portfolio distribution
ax = axes[0]
counts = [band_summary.loc[b, 'count'] for b in band_order]
bars = ax.bar(band_order, counts, color=band_colors, edgecolor='white', linewidth=1.5)
for bar, cnt in zip(bars, counts):
    pct = cnt / len(risk_df) * 100
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
            f'{cnt:,}\n({pct:.1f}%)', ha='center', va='bottom',
            fontsize=10, fontweight='bold')
ax.set_title('Portfolio Distribution by Risk Band', fontweight='bold')
ax.set_ylabel('Number of Customers')
ax.set_ylim(0, max(counts) * 1.2)

# Default rate per band
ax = axes[1]
def_rates = [band_summary.loc[b, 'default_rate'] for b in band_order]
bars2 = ax.bar(band_order, def_rates, color=band_colors, edgecolor='white', linewidth=1.5)
for bar, rate in zip(bars2, def_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{rate:.1%}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Observed Default Rate by Risk Band', fontweight='bold')
ax.set_ylabel('Default Rate')
ax.set_ylim(0, max(def_rates) * 1.3)

# Probability histogram by band
ax = axes[2]
for band in band_order:
    subset = risk_df.loc[risk_df['risk_band'] == band, 'probability']
    ax.hist(subset, bins=30, alpha=0.65, color=COLORS[band], label=band)
ax.axvline(x=0.20, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
ax.axvline(x=0.50, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
ax.set_xlabel('Predicted Probability of Default')
ax.set_ylabel('Count')
ax.set_title('Probability Distribution by Risk Band', fontweight='bold')
ax.legend()

plt.suptitle('Risk Band Analysis — XGBoost Champion Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / '21_risk_band_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Final Decision & Policy Reference

### Recommended Threshold: t = 0.30

**Rationale:**
- Recall ≥ 0.70 — catches 70%+ of defaulters before approval, protecting the portfolio.
- Total business cost is materially lower than at t = 0.50 (see cost analysis above).
- Approval rate ~46.6% — lower than naive 0.50 but operationally viable; avoids overwhelming analyst capacity.
- Defensible under **SR 11-7**: the threshold selection is documented with a cost-model justification.

### Risk Band Policy
See **`docs/threshold_policy.md`** for the full business policy document used by the credit operations team.

> **Day 12 preview:** SHAP explainability — making XGBoost SR 11-7 compliant by generating per-decision explanations.

In [ ]:
RECOMMENDED_T = 0.30
y_pred_final = (y_probs >= RECOMMENDED_T).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_final).ravel()

cost_30 = fn * FN_COST + fp * FP_COST
cost_50 = int(cost_df.loc[0.50, '_cost'])

print('=== OPERATIONAL THRESHOLD: t = 0.30 ===')
print(f'  True  Positives (caught defaults)  : {tp:>5,}')
print(f'  True  Negatives (approved good)    : {tn:>5,}')
print(f'  False Positives (wrongly declined) : {fp:>5,}')
print(f'  False Negatives (missed defaults)  : {fn:>5,}')
print()
print(f'  Precision : {precision_score(y_test, y_pred_final):.4f}')
print(f'  Recall    : {recall_score(y_test, y_pred_final):.4f}')
print(f'  F1        : {f1_score(y_test, y_pred_final):.4f}')
print(f'  AUC-ROC   : {roc_auc_score(y_test, y_probs):.4f}')
print()
print(f'  Business Cost @ t=0.30 : ${cost_30:>10,.0f}')
print(f'  Business Cost @ t=0.50 : ${cost_50:>10,.0f}')
print(f'  Cost savings vs default : ${cost_50 - cost_30:>10,.0f}')
print()
print('Policy reference : docs/threshold_policy.md')
print('Day 11 COMPLETE — Model is now a Decision Engine.')